# Quafing analysis environment

This notebook provides a graphical user interface for Questionaire anlysis using the quafing package. NOTE: this supports common workflows, but does not provide access to all features supported by the `quafing` library. For specialized use cases we refer to `quafing`'s documentation and direct use of the library.


In [1]:
#import necessary packages
import os 
#import quafing as q


import ipywidgets as widgets
from ipywidgets import interact, interact_manual, fixed
from ipywidgets import HBox, VBox
from IPython.display import display
#from ipython.display import display

In [2]:
os.chdir('../')
print(os.getcwd())
import quafing as q

/Users/mwgrootes/Projects/REPOS/SDCCA/quafing


## Data loading
Data for analysis can be loaded in to by specifying a filepath (see below) 


### Standard format
Currently data input is supported from spreadsheet type files (.xls, .xlsx, .xlsm, .xlsb, .odf, .ods, .odt). `quafing` assumes columnar data with meta data on the columns located on the same sheet. The standard format corresponds to (all columns and row are 0-indexed):

- Data and metadata are located on sheet 0.
- Row 0 contains the column type (see below)
- Row 1 contains the number of the asociated question
- Row 2 (header row) contains the column names
- Data starts on row 3
- No rows (read 0) are skipped at the end

standard row types (denoted by single str characters) are:

    e: excluded
    g: group by this column
    c: continuous variable
    u: unordered discrete
    o: ordered discrete
    b: binary
    
It should be emphasized that the user can depart from this standard. As long as the basic format of columnar data with metadata for each column is maintained, the actual inddicees of the rows can be changed. Similarly a different row type schema can be used, albeit preferably string based. However, such alterations require additional specification in quafings functions, while the default values are configired to support the standard schema

In [3]:
filepath = '/Users/mwgrootes/Projects/DATA/SDCCA/test/quafing_toy.xlsx' #please specify absolute path to data

In [4]:
class Quaffing_ui(object):
    def __init__(self,datapath):
        self.datapath = datapath
        self.rawmetadata = None
        self.rawdata = None
        self.all_columns = None
        self.select_mode = None
        self.selected_columns = None
        self.deselected_columns = None
        self.group_by_col = None
        self.analysis_columns = None

    def load(self):
        rawmetadata,rawdata = q.load(self.datapath) #no further arguments necessary in this case
        self.rawmetadata = rawmetadata
        self.rawdata = rawdata
        self.all_columns = rawmetadata['ColNames']

    #Step 1 select columns for analysis and column to group by
    def select_column_and_group(self):
        columns_select_mode_widget = widgets.RadioButtons(
            options=['select columns', 'deselect columns'],
            value='select columns', #defaults to selection of columns
            layout={'width':'max-content'},
            description='column selection mode',
            disabled=False
        )
        columns_select_list_widget = widgets.SelectMultiple(
            options=self.all_columns,
            value=[self.all_columns[0]],
            description='Columns',
            disabled=False
        )
        columns_select_groupby_widget_2 = widgets.Dropdown()
        columns_cont_dens_tab_widget = widgets.Tab()
        contden_cboxes = []
        contden_dboxes = [] 

        def update(_):
            selm = columns_select_mode_widget.value
            self.select_mode = selm
            sl = list(columns_select_list_widget.value)
            if selm == "deselect columns":
                ll = list([ent for ent in self.all_columns if ent not in sl ])
            else:
                ll = sl
            columns_select_groupby_widget_2.options=ll
            groupby_val = columns_select_groupby_widget_2.value
            if ll is not None:
                analysis_columns = [ent for ent in ll if ent != groupby_val]
            else:
                analysis_columns= None
            self.analysis_columns = analysis_columns
            if analysis_columns is not None:
                contden_cboxes = [widgets.Checkbox(description=f'{col} is continuos variable') for col in analysis_columns]
                contden_dboxes = [widgets.Dropdown(description='density method to apply') for col in analysis_columns]
                children = [widgets.VBox([contden_cboxes[i], contden_dboxes[i]]) for i in range(len(analysis_columns))]
                columns_cont_dens_tab_widget.children = children
                columns_cont_dens_tab_widget.titles = [col for col in self.analysis_columns]
            
            

        columns_select_list_widget.observe(update)
        columns_select_mode_widget.observe(update)
        columns_select_groupby_widget_2.observe(update)
        columns_cont_dens_tab_widget.observe(update)

        def selected_columns(select_mode,columns):
            mode = select_mode
            cols = columns
            if mode == "deselect columns":
                collist = list([ent for ent in self.all_columns if ent not in cols ])
                dcollist = cols
            else:
                collist = cols
                dcollist = list([ent for ent in self.all_columns if ent not in cols])
            return collist, dcollist    

        def select_col_ui_func_2(**kwargs):
            sel_columns, dsel_columns = selected_columns(select_mode,columns)
            print (sel_columns)
            print (dsel_columns)
            print (groupby)
            print (cb[0])
            print(db[0])
            self.selected_columns = sel_columns
            self.deselected_columns = dsel_columns
            self.group_by_col = groupby
            return self.selected_columns,self.deselected_columns,self.group_by_col

        def select_col_ui_func(select_mode,columns,groupby):
            sel_columns, dsel_columns = selected_columns(select_mode,columns)
            print (sel_columns)
            print (dsel_columns)
            print (groupby)
            self.selected_columns = sel_columns
            self.deselected_columns = dsel_columns
            self.group_by_col = groupby
            return self.selected_columns,self.deselected_columns,self.group_by_col
        
        ui_selctrl = VBox([HBox([columns_select_mode_widget, columns_select_list_widget, columns_select_groupby_widget_2]),HBox([columns_cont_dens_tab_widget])])
        selctrl = widgets.interactive_output(select_col_ui_func,{'select_mode':columns_select_mode_widget, 'columns':columns_select_list_widget,'groupby':columns_select_groupby_widget_2})
        display(ui_selctrl,selctrl)
"""
    def select_cont_density(self):
        #columns_select_continuous_widget = widgets.SelectMultiple(
        #    description = 'Select columns with continuos data:',
        #    diabled=False
        #)
        children = [ widgets.VBox([widgets.Checkbox(description=f'{col} is continuous variable'), widgets.Dropdown(description='density method to apply')]) for col in self.analysis_columns]
        tab = widgets.Tab()
        tab.children = children
        tab.titles = [col for col in self.analysis_columns]
        def select_cont_density_ui_func(selw):
            print (test)
        ui_contdenctrl = tab
        def select_cont_density_ui_func
        #contdenctrl = widgets.interactive_output(select_cont_density_ui_func, {"selw":tab})
        display(ui_contdenctrl)#,contdenctrl)
"""


    






'\n    def select_cont_density(self):\n        #columns_select_continuous_widget = widgets.SelectMultiple(\n        #    description = \'Select columns with continuos data:\',\n        #    diabled=False\n        #)\n        children = [ widgets.VBox([widgets.Checkbox(description=f\'{col} is continuous variable\'), widgets.Dropdown(description=\'density method to apply\')]) for col in self.analysis_columns]\n        tab = widgets.Tab()\n        tab.children = children\n        tab.titles = [col for col in self.analysis_columns]\n        def select_cont_density_ui_func(selw):\n            print (test)\n        ui_contdenctrl = tab\n        def select_cont_density_ui_func\n        #contdenctrl = widgets.interactive_output(select_cont_density_ui_func, {"selw":tab})\n        display(ui_contdenctrl)#,contdenctrl)\n'

In [5]:
qui = Quaffing_ui(filepath)

In [6]:
qui.load()

In [7]:
qui.select_column_and_group()

Output()